# News Topic Classifier Using BERT

This project builds a news classification model using the AG News dataset.  
A pretrained BERT model is fine-tuned to classify news headlines into four categories:

- World
- Sports
- Business
- Sci/Tech

In [1]:
from datasets import load_dataset
from transformers import BertTokenizer
import torch


## Load AG News Dataset

The dataset is loaded using the Hugging Face datasets library.  
It contains news headlines labeled into four categories.

In [2]:
dataset = load_dataset("ag_news")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
dataset["train"][0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [4]:
dataset["train"].features

{'text': Value('string'),
 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}

## Load BERT Tokenizer

## Tokenization

Text data must be converted into numerical tokens before being fed into BERT.  
The BERT tokenizer converts news headlines into input IDs and attention masks.

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

## Test Tokenizer

In [6]:
example = dataset["train"][0]["text"]

tokens = tokenizer(example)

print(tokens)

{'input_ids': [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## Tokenize Entire Dataset

In [7]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

## Check Tokenized Dataset

In [8]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

## Reducing Size

In [9]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(300))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

## Removing Unnecessary Columns

In [10]:
small_train = small_train.remove_columns(["text"])
small_test = small_test.remove_columns(["text"])

## Set format

In [11]:
small_train.set_format("torch")
small_test.set_format("torch")

##  Load Pretrained BERT Model

We load the pretrained BERT model and adapt it for a classification task with four output labels.

In [12]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", 
    num_labels=4,
    local_files_only=False  
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1884.94it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tr

## Define Training Arguements

In [13]:

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1
)

## Create Trainer

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
)


## Model Training

The model is fine-tuned on the training dataset using the Trainer API.  
Fine-tuning allows BERT to learn patterns specific to news topic classification.

In [15]:
trainer.train()

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.58s/it]


TrainOutput(global_step=150, training_loss=0.6907852172851563, metrics={'train_runtime': 2412.301, 'train_samples_per_second': 0.124, 'train_steps_per_second': 0.062, 'total_flos': 78934734028800.0, 'train_loss': 0.6907852172851563, 'epoch': 1.0})


## Model Evaluation

The trained model is evaluated using:

- Accuracy
- F1 Score

These metrics help measure the classification performance.

In [24]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


predictions = trainer.predict(small_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [23]:
preds

array([1, 2, 1, 2, 3, 2, 2, 2, 3, 2, 3, 1, 1, 0, 3, 3, 0, 1, 0, 3, 0, 2,
       1, 1, 0, 2, 2, 2, 2, 2, 2, 3, 3, 2, 3, 2, 1, 2, 2, 1, 1, 0, 0, 3,
       0, 1, 2, 1, 2, 1, 3, 0, 2, 2, 1, 2, 0, 3, 2, 1, 2, 1, 1, 2, 1, 3,
       2, 1, 1, 2, 0, 1, 0, 0, 0, 1, 1, 3, 3, 2, 0, 1, 1, 0, 1, 3, 1, 2,
       3, 2, 1, 2, 2, 1, 2, 3, 2, 3, 3, 3])

In [22]:
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8300
F1 Score: 0.8289


## Save Model

In [25]:
model.save_pretrained("news_bert_model")
tokenizer.save_pretrained("news_bert_model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


('news_bert_model\\tokenizer_config.json', 'news_bert_model\\tokenizer.json')

## Gradio Demo Interface

A Gradio interface is created to allow users to input a news headline and receive the predicted category from the trained model.

### Label Mapping

In [26]:
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

### Build Gradio Function

In [19]:
import torch

def predict_news(text):
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    outputs = model(**inputs)
    
    predicted_class = torch.argmax(outputs.logits).item()
    
    return labels[predicted_class]

### Build Gradio Interface

In [20]:
import gradio as gr

interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Enter a news headline and the model will predict its category."
)

### Launch the app

In [21]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Conclusion

In this project, a news topic classification system was developed using a pretrained BERT model. 
The AG News dataset was used to train and evaluate the model. The text data was first tokenized 
using the BERT tokenizer and then fine-tuned using a sequence classification architecture.

The model was evaluated using accuracy and F1-score metrics to measure its classification 
performance across four categories: World, Sports, Business, and Sci/Tech.

Finally, an interactive interface was built using Gradio to allow users to input news headlines 
and receive predicted categories in real time.

This project demonstrates how transformer-based models can be effectively applied to natural 
language processing tasks such as text classification.